# Modelagem Preditiva por Município

**Tech Challenge — Fase 3 | FIAP AI Scientist**

## 1. Objetivo do projeto

Este notebook documenta os experimentos de modelagem preditiva para identificar municípios com maior risco de não atingir a meta educacional definida no projeto. A análise compara modelos, avalia seu desempenho em dados separados e investiga os atributos mais relevantes para as previsões.

Os resultados representam associações preditivas e não devem ser interpretados como evidência de causalidade.

## 2. Dados e definição do alvo

A tabela analítica é construída a partir das tabelas de metas, indicadores, municípios e redes disponíveis no projeto. A unidade de análise é o município, conforme a tabela preparada pelo pipeline.

A variável-alvo é `nao_atingiu_meta`: o valor 1 representa municípios que não atingiram a meta, enquanto o valor 0 representa municípios que atingiram a meta.

A configuração `COLUNAS_T2`, definida em `src/config.py`, especifica os atributos preditores. A variável-alvo e os identificadores não são utilizados como atributos de entrada.

## 3. Preparação dos dados e prevenção de vazamento

A tabela municipal é preparada utilizando as funções já existentes no projeto. A separação entre treino, validação e teste é realizada pelo módulo `src/modeling/separacao.py`, preservando a separação dos municípios entre os conjuntos.

O pré-processamento dos atributos é integrado ao pipeline do modelo. As decisões finais de modelagem não devem ser baseadas no desempenho do conjunto de teste.

In [1]:
from src.preprocessing.loader import carregar_tabela
from src.preprocessing.tabelas import construir_tabela_municipio
from src.modeling.separacao import separar_por_municipio
from src.config import COLUNAS_T2, ALVO_MUNICIPIO, SEED
import pandas as pd

metas = carregar_tabela('fact_meta_resultado_municipio')
indicadores = carregar_tabela('fact_indicador_municipio')
municipios = carregar_tabela('dim_municipio')
redes = carregar_tabela('dim_rede')

tabela = construir_tabela_municipio(
    metas, indicadores, municipios, redes
)
treino, validacao, teste = separar_por_municipio(tabela)

print('Total de registros:', len(tabela.atributos))
print('Treino:', len(treino.atributos))
print('Validação:', len(validacao.atributos))
print('Teste:', len(teste.atributos))
print('Atributos utilizados:', COLUNAS_T2)

Total de registros: 5226
Treino: 3135
Validação: 1045
Teste: 1046
Atributos utilizados: ColunasTrilha(categoricas=('sigla_uf', 'nome_regiao'), numericas=('capital_uf', 'idhm', 'idhm_educacao', 'idhm_renda', 'idhm_longevidade', 'taxa_alfabetizacao_2023', 'media_portugues_2023', 'meta_indicador_2024', 'folga_2023'), proibidas=('atingiu_meta', 'nao_atingiu_meta', 'gap_pontos', 'id_municipio', 'rede', 'taxa_alfabetizacao_2024', 'media_portugues_2024', 'percentual_participacao_2024', 'nivel_alfabetizacao', 'valid_from', 'valid_to', 'is_current'), prefixos_proibidos=('proporcao_aluno_nivel_',))


In [2]:
print('Distribuição do alvo:')
distribuicao_alvo = tabela.alvo.value_counts(normalize=False).sort_index()
display(distribuicao_alvo.to_frame(name='quantidade'))

print('\nProporção por classe (%):')
display((tabela.alvo.value_counts(normalize=True).sort_index() * 100).round(2))

Distribuição do alvo:


,quantidade
nao_atingiu_meta,
0,2782
1,2444



Proporção por classe (%):


,proportion
nao_atingiu_meta,
0,53.23
1,46.77


## 4. Modelo de referência e validação cruzada

A comparação utiliza validação cruzada agrupada por município, com cinco dobras. O pré-processamento é ajustado dentro de cada dobra, utilizando apenas os dados de treino daquela dobra.

O Dummy é o modelo de referência. A regressão logística é avaliada pela AUC-ROC, que mede a capacidade de discriminar municípios que atingiram ou não a meta.

In [3]:
from src.modeling.protocolo import executar_baselines

resultados_baseline = executar_baselines(
    trilha=COLUNAS_T2,
    tabela=treino,
    n_dobras=5
)

for nome_modelo, resultado in resultados_baseline.items():
    print(f'\nModelo: {nome_modelo}')
    for metrica, valor in resultado.items():
        print(f'  {metrica}: {valor}')


Modelo: dummy
  auc_roc_por_dobra: [0.5, 0.5, 0.5, 0.5, 0.5]
  auc_roc_media: 0.5
  auc_roc_desvio: 0.0
  auc_roc_treino_media: 0.5

Modelo: regressao_logistica
  auc_roc_por_dobra: [0.7933746522664049, 0.7773126403920769, 0.7925072927928847, 0.7836603036346615, 0.7530187398432517]
  auc_roc_media: 0.7799747257858559
  auc_roc_desvio: 0.014723577189874994
  auc_roc_treino_media: 0.7903079140351259


### Interpretação

A regressão logística obteve AUC-ROC média de aproximadamente 0,780 (desvio-padrão de 0,015), superando o Dummy, cuja AUC-ROC foi 0,500 em todas as dobras. A AUC-ROC média no treino foi 0,790, próxima à média de validação cruzada. Esses resultados sustentam a regressão logística como candidata principal, sem eliminar a necessidade de avaliação final no conjunto de teste.

## 5. Comparação de modelos

Além da regressão logística, foram avaliadas duas configurações de Random Forest. A comparação considera a AUC-ROC média na validação cruzada agrupada por município.

| Modelo | AUC-ROC média | AUC-ROC média no treino |
|---|---:|---:|
| Regressão logística | 0,780 | 0,790 |
| Random Forest padrão | 0,755 | 1,000 |
| Random Forest regularizado | 0,769 | 0,836 |

Os resultados exploratórios favorecem a regressão logística. O Random Forest padrão apresentou AUC-ROC de treino igual a 1,000, com desempenho inferior na validação cruzada, o que sugere sobreajuste. A versão regularizada reduziu essa diferença, mas não superou a regressão logística.

**Nota:** os valores acima são resultados dos experimentos exploratórios já realizados. A reprodução abaixo permite conferir os resultados usando os módulos do projeto.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from src.modeling.pipeline_modelo import montar_pipeline
from src.modeling.validacao_cruzada import validar_com_cv
from src.config import SEED

# Random Forest padrão
rf_padrao = montar_pipeline(
    trilha=COLUNAS_T2,
    estimador=RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        class_weight='balanced',
        n_jobs=-1
    ),
    escalonar=False
)

# Random Forest regularizado
rf_regularizado = montar_pipeline(
    trilha=COLUNAS_T2,
    estimador=RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=10,
        random_state=SEED,
        class_weight='balanced',
        n_jobs=-1
    ),
    escalonar=False
)

resultado_rf_padrao = validar_com_cv(rf_padrao, treino, n_dobras=5)
resultado_rf_regularizado = validar_com_cv(
    rf_regularizado, treino, n_dobras=5
)

print('Random Forest padrão:')
print(resultado_rf_padrao)
print('\nRandom Forest regularizado:')
print(resultado_rf_regularizado)

Random Forest padrão:
{'auc_roc_por_dobra': [0.7584990181639666, 0.7666122115580968, 0.7736378286856653, 0.7544059587284789, 0.7212165470141731], 'auc_roc_media': 0.7548743128300762, 'auc_roc_desvio': 0.01808895153320059, 'auc_roc_treino_media': 1.0}

Random Forest regularizado:
{'auc_roc_por_dobra': [0.7704958271968583, 0.7780171533591994, 0.7874788356010689, 0.7503052627294559, 0.7577602698866559], 'auc_roc_media': 0.7688114697546478, 'auc_roc_desvio': 0.013421333281354256, 'auc_roc_treino_media': 0.8365379221508583}


## 6. Avaliação do modelo finalista

A regressão logística foi selecionada com base na comparação por validação cruzada agrupada por município. Nesta etapa, o modelo é ajustado com os dados de treino e avaliado no conjunto de validação. O conjunto de teste permanece reservado para a avaliação final, sem ser utilizado na escolha do modelo.

In [5]:
from sklearn.linear_model import LogisticRegression
from src.modeling.pipeline_modelo import montar_pipeline
from src.evaluation.metricas import calcular_metricas
from src.config import COLUNAS_T2, SEED

# Treinar a regressão logística com os dados de treino
modelo_finalista = montar_pipeline(
    trilha=COLUNAS_T2,
    estimador=LogisticRegression(
        max_iter=200,
        random_state=SEED
    ),
    escalonar=True
)

modelo_finalista.fit(treino.atributos, treino.alvo)

# Avaliar no conjunto de validação
probabilidades_validacao = modelo_finalista.predict_proba(
    validacao.atributos
)[:, 1]

metricas_validacao = calcular_metricas(
    validacao.alvo,
    probabilidades_validacao,
    limiar=0.5
)

print("Métricas de validação:")
print(metricas_validacao)

Métricas de validação:
{'auc_roc': 0.751917631917632, 'auc_pr': 0.7317767953952627, 'precisao': 0.6750524109014675, 'recall': 0.6571428571428571, 'f1': 0.6659772492244054, 'matriz_confusao': {'verdadeiro_negativo': 400, 'falso_positivo': 155, 'falso_negativo': 168, 'verdadeiro_positivo': 322}}


### Interpretação da validação

A avaliação no conjunto de validação permite analisar a capacidade de discriminação do modelo e o equilíbrio entre precisão e recall, considerando o limiar de classificação de 0,5.

A AUC-ROC e a AUC-PR avaliam a qualidade da ordenação das probabilidades previstas. Precisão, recall e F1 complementam essa análise considerando as classificações obtidas no limiar escolhido.

As métricas de validação são utilizadas para compreender o comportamento do modelo antes da avaliação final no conjunto de teste.

## 7. Avaliação final no conjunto de teste

Após a seleção da regressão logística com base na validação cruzada, o modelo é avaliado no conjunto de teste, que permaneceu separado durante as etapas de seleção.

As métricas abaixo representam o desempenho do modelo nos municípios reservados para essa avaliação final.

In [6]:
# Ajustar o modelo finalista com treino e validação
from sklearn.linear_model import LogisticRegression
from src.modeling.pipeline_modelo import montar_pipeline
from src.evaluation.metricas import calcular_metricas
from src.config import COLUNAS_T2, SEED
import pandas as pd

treino_validacao_atributos = pd.concat(
    [treino.atributos, validacao.atributos],
    axis=0,
    ignore_index=True
)

treino_validacao_alvo = pd.concat(
    [treino.alvo, validacao.alvo],
    axis=0,
    ignore_index=True
)

modelo_teste = montar_pipeline(
    trilha=COLUNAS_T2,
    estimador=LogisticRegression(
        max_iter=200,
        random_state=SEED
    ),
    escalonar=True
)

modelo_teste.fit(
    treino_validacao_atributos,
    treino_validacao_alvo
)

probabilidades_teste = modelo_teste.predict_proba(
    teste.atributos
)[:, 1]

metricas_teste = calcular_metricas(
    teste.alvo,
    probabilidades_teste,
    limiar=0.5
)

print("Métricas finais no teste:")
print(metricas_teste)

Métricas finais no teste:
{'auc_roc': 0.7964302551711945, 'auc_pr': 0.7680861099156716, 'precisao': 0.6896551724137931, 'recall': 0.6694560669456067, 'f1': 0.6794055201698513, 'matriz_confusao': {'verdadeiro_negativo': 424, 'falso_positivo': 144, 'falso_negativo': 158, 'verdadeiro_positivo': 320}}


### Interpretação dos resultados no teste

A avaliação final da regressão logística apresentou os seguintes resultados:

- **AUC-ROC:** 0,796
- **AUC-PR:** 0,768
- **Precisão:** 0,690
- **Recall:** 0,669
- **F1-score:** 0,679

A AUC-ROC indica a capacidade do modelo de distinguir municípios que atingiram a meta daqueles que não a atingiram. A precisão representa a proporção de municípios classificados como não atingindo a meta que realmente pertencem a essa classe. O recall representa a proporção dos municípios que não atingiram a meta que foram identificados pelo modelo.

Com o limiar de classificação de 0,5, o modelo identificou 320 municípios que não atingiram a meta, deixou de identificar 158 municípios dessa classe e classificou incorretamente 144 municípios que atingiram a meta.

Esses resultados descrevem o desempenho no conjunto de teste e não garantem a mesma performance em dados futuros.


## 8. Importância das variáveis

A importância das variáveis é analisada por permutação no conjunto de validação, utilizando a AUC-ROC como métrica. O método avalia quanto o desempenho do modelo se altera quando os valores de um atributo são embaralhados.

Uma redução maior na AUC-ROC após a permutação indica maior contribuição daquele atributo para a discriminação realizada pelo modelo. Essa análise descreve associações preditivas e não estabelece relações de causa e efeito.

In [7]:
from sklearn.inspection import permutation_importance

# Medir a importância dos atributos na validação
importancia = permutation_importance(
    modelo_finalista,
    validacao.atributos,
    validacao.alvo,
    scoring='roc_auc',
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1
)

tabela_importancia = pd.DataFrame({
    'atributo': validacao.atributos.columns,
    'importancia_media': importancia.importances_mean,
    'desvio_padrao': importancia.importances_std
}).sort_values(
    'importancia_media',
    ascending=False
).reset_index(drop=True)

display(tabela_importancia.round(4))

,atributo,importancia_media,desvio_padrao
0,sigla_uf,0.1697,0.0104
1,meta_indicador_2024,0.0863,0.0072
2,media_portugues_2023,0.0835,0.0071
3,taxa_alfabetizacao_2023,0.0441,0.0074
4,nome_regiao,0.0333,0.0062
5,folga_2023,0.0265,0.0048
6,idhm_educacao,0.0032,0.0020
7,idhm_longevidade,0.0029,0.0009
8,idhm,0.0014,0.0005
9,capital_uf,0.0004,0.0003


### Interpretação da importância das variáveis

Na análise por permutação, os atributos com maior importância média foram:

1. `sigla_uf`
2. `meta_indicador_2024`
3. `media_portugues_2023`
4. `taxa_alfabetizacao_2023`
5. `nome_regiao`

Esses resultados indicam que a localização geográfica, a meta educacional e os indicadores educacionais anteriores contribuíram para a capacidade do modelo de distinguir municípios que atingiram ou não a meta.

A importância por permutação mede a contribuição preditiva de cada atributo no modelo avaliado. Portanto, não demonstra que esses atributos causem o resultado educacional. A importância também pode variar conforme os dados utilizados e a relação entre os atributos.